In [11]:
import re

def validar_cadastro(nome, email, idade, senha):
    erros = []

    if len(nome.strip()) < 3:
        erros.append("Nome inválido")

    if "@" not in email or "." not in email.split("@")[-1]:
        erros.append("Email inválido")

    if idade < 18 or idade > 120:
        erros.append("Idade inválida")

    if len(senha) < 8:
        erros.append("Senha muito curta")
    elif not re.search(r"[A-Za-z]", senha) or not re.search(r"\d", senha):
        erros.append("Senha precisa conter letra e número")

    return len(erros) == 0, erros

In [12]:
validar_cadastro("Rodrigo", "rodrigo@email.com", 42, "ABC123456")

(True, [])

In [13]:
import unittest

class TestCadastro(unittest.TestCase):

    def test_cadastro_valido(self):
        ok, erros = validar_cadastro("Ana", "ana@email.com", 30, "senha123")
        self.assertTrue(ok)

    def test_nome_invalido(self):
        ok, erros = validar_cadastro("An", "ana@email.com", 30, "senha123")
        self.assertFalse(ok)

unittest.main(argv=[''], exit=False)

..
----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK


📌 Desafio

Crie 3 novos testes para a função validar_cadastro:

✔ Um caso extremo
✔ Um caso inválido
✔ Um caso que valide regra de negócio

In [14]:
import re
from dataclasses import dataclass
from typing import List, Tuple

def validar_cadastro(nome: str, email: str, idade: int, senha: str) -> Tuple[bool, List[str]]:
    """
    Regras:
    - nome: mínimo 3 caracteres (ignorando espaços)
    - email: precisa ter '@' e '.' após o @
    - idade: entre 18 e 120
    - senha: mínimo 8, precisa ter letra e número
    """
    erros = []

    if not isinstance(nome, str) or len(nome.strip()) < 3:
        erros.append("Nome inválido: mínimo 3 caracteres.")

    if not isinstance(email, str) or "@" not in email:
        erros.append("Email inválido: precisa conter @.")
    else:
        dominio = email.split("@")[-1]
        if "." not in dominio:
            erros.append("Email inválido: domínio precisa conter '.'.")

    if not isinstance(idade, int) or idade < 18 or idade > 120:
        erros.append("Idade inválida: deve estar entre 18 e 120.")

    if not isinstance(senha, str) or len(senha) < 8:
        erros.append("Senha inválida: mínimo 8 caracteres.")
    else:
        if not re.search(r"[A-Za-z]", senha) or not re.search(r"\d", senha):
            erros.append("Senha fraca: precisa ter letra e número.")

    return (len(erros) == 0, erros)

In [15]:
casos = [
    ("Rodrigo", "rodrigo@email.com", 42, "abc12345"),
    ("Ro", "ro@email.com", 30, "abc12345"),
    ("Ana", "anaemail.com", 30, "abc12345"),
    ("Ana", "ana@emailcom", 30, "abc12345"),
    ("Ana", "ana@email.com", 15, "abc12345"),
    ("Ana", "ana@email.com", 30, "abcdefgh"),
]

for i, c in enumerate(casos, 1):
    ok, erros = validar_cadastro(*c)
    print(f"Caso {i}: ok={ok} | erros={erros}")

Caso 1: ok=True | erros=[]
Caso 2: ok=False | erros=['Nome inválido: mínimo 3 caracteres.']
Caso 3: ok=False | erros=['Email inválido: precisa conter @.']
Caso 4: ok=False | erros=["Email inválido: domínio precisa conter '.'."]
Caso 5: ok=False | erros=['Idade inválida: deve estar entre 18 e 120.']
Caso 6: ok=False | erros=['Senha fraca: precisa ter letra e número.']


## Como QA pensa aqui?
- Cada regra precisa ser testada:
  - **fluxo feliz**
  - **fluxos inválidos**
  - **casos extremos**
- Não basta dizer "deu erro". QA registra:
  - **resultado atual**
  - **resultado esperado**
  - **evidência** (log/print/teste automatizado)

In [16]:
def validar_email_bugado(email: str) -> bool:
    # BUG proposital: aceita 'a@b' como válido (sem ponto no domínio)
    return isinstance(email, str) and ("@" in email)

testes_email = ["ana@email.com", "ana@emailcom", "anaemail.com", "a@b"]
for e in testes_email:
    print(e, "=>", validar_email_bugado(e))

ana@email.com => True
ana@emailcom => True
anaemail.com => False
a@b => True


DESAFIO DO BUG ESCONDIDO

In [17]:
def calcular_desconto(valor: float, cupom: str) -> float:
    """
    Regras:
    - Cupom 'QA10' dá 10% de desconto
    - Cupom 'QA20' dá 20% de desconto, somente se valor >= 200
    - Cupom inválido não dá desconto
    BUG proposital: implementação tem um erro de regra
    """
    if cupom == "QA10":
        return valor * 0.9
    if cupom == "QA20":
        if valor >= 200:  # Condição adicionada para aplicar o desconto de QA20
            return valor * 0.8
        else:
            return valor # Não aplica desconto se valor < 200
    return valor

In [ ]:
# @title Texto de título padrão
variable_name = "" # @param {"type":"string"}
def testar_calcular_desconto():
    # Teste 1: QA20 com valor < 200 (deve NÃO aplicar desconto)
    assert calcular_desconto(150, "QA20") == 150, "Bug: Desconto aplicado para valor < 200"

    # Teste 2: QA20 com valor >= 200 (deve aplicar desconto)
    assert calcular_desconto(250, "QA20") == 200, "Desconto não aplicado para valor >= 200"

    # Teste 3: QA10 com qualquer valor (deve aplicar 10% de desconto)
    assert calcular_desconto(100, "QA10") == 90, "Desconto QA10 não aplicado corretamente"

    # Teste 4: Cupom inválido (deve não aplicar desconto)
    assert calcular_desconto(100, "INVALIDO") == 100, "Desconto aplicado para cupom inválido"

    # Teste 5: QA20 com valor = 200 (deve aplicar desconto)
    assert calcular_desconto(200, "QA20") == 160, "Desconto não aplicado para valor = 200"

    print("Todos os testes passaram!")

testar_calcular_desconto()

# O problema foi resolvido corrigindo o return final e ajustando a lógica de cada cupom.
# Criei testes para garantir que:
# - QA10 sempre aplica 10% de desconto
# - QA20 aplica 20% apenas se o valor >= 200
# - Cupons inválidos não aplicam desconto
# Todos os testes passam, garantindo que a função segue as regras corretamente.